##DBTC-1145

In [0]:
query = """
    SELECT DISTINCT d.businessdate
    FROM `p-cdh`.derivative.dexposure d
    INNER JOIN `p-cdh`.reference.ref_calendar c 
    ON d.businessdate = c.CalendarDate
    WHERE c.IsBusinessDay = 0 OR c.isholiday = 1
"""
df = spark.sql(query)

delete_dates = [row.businessdate for row in df.collect()]
print(delete_dates)

In [0]:
for delete_date in delete_dates:
    query = f"""
    DELETE FROM `p-cdh`.derivativearchive.dexposure
    WHERE BusinessDate = '{delete_date}'
    """
    spark.sql(query)

## RBTC-153

In [0]:
%sql

CREATE TABLE `p-cdh`.reference.modifications_pee_20250217
AS SELECT * FROM `d-cdh`.reference.modifications_pee_20250217 

## BOGCCR-394

In [0]:
%sql
drop table if exists `d-cdh`.primaryactive.exposureBOGCCR394_final;

In [0]:
%sql
create table `d-cdh`.primaryactive.exposureBOGCCR394_final as 
SELECT DISTINCT B.*
FROM (
select count(1) c, businessdate, ExposureID, CDI
from `p-cdh`.primaryactive.exposure 
where sourcesystem='profits' 
group by businessdate, ExposureID, CDI
having count(1)>1) AS A
inner join `p-cdh`.primaryactive.exposure B on a.ExposureID=b.ExposureID and a.businessdate=b.businessdate and a.CDI=b.CDI;

DELETE FROM `p-cdh`.primaryactive.exposure B
WHERE EXISTS (
    SELECT 1
    FROM (
        SELECT businessdate, ExposureID, CDI
        FROM `p-cdh`.primaryactive.exposure
        WHERE sourcesystem = 'profits'
        GROUP BY businessdate, ExposureID, CDI
        HAVING COUNT(1) > 1
    ) AS A
    WHERE a.ExposureID = B.ExposureID
      AND a.businessdate = B.businessdate
      AND a.CDI = B.CDI
);

INSERT INTO `p-cdh`.primaryactive.exposure
SELECT * FROM  `d-cdh`.primaryactive.exposureBOGCCR394_final;

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

